# Victoria driving rules chatbot (RAG)

A chatbot that answers international students' questions about driving in Victoria, Australia, using only the knowledge base you built. Each step below is one cell, with a short explanation of what it does.

**The idea (RAG = retrieval-augmented generation):**
1. Split the knowledge base into chunks and turn each chunk into a list of numbers (a vector) that captures its meaning.
2. When someone asks a question, turn the question into a vector too and find the chunks whose vectors are closest.
3. Give those chunks, plus the question, to a free AI model (Groq first, Google Gemini as backup) and tell it to answer **only** from them.

**Before you start**
- Keep `data/integrated_kb_chunks.json` next to this notebook and run the notebook from the repo root (on Google Colab: upload the whole folder).
- Steps 1 to 4 need no API key. Step 5 needs at least one free AI key. Groq is the easiest: https://console.groq.com/keys (no card)
- Step 8 (the evaluation) also needs `data/evaluation_questions.json`.
- Run the cells from top to bottom.

## Step 0. Install the libraries (run once)

In [ ]:
%pip install -q sentence-transformers chromadb openai ipywidgets

## Step 1. Load the knowledge base chunks

`integrated_kb_chunks.json` was made from your integrated Word file. Each chunk is one piece of it (a paragraph group, a table, or a sign with its meaning) together with its heading, the "Applies to" line and the source.

In [ ]:
import json, hashlib, os
from collections import Counter
from pathlib import Path

CHUNK_FILE = Path("data/integrated_kb_chunks.json")   # run the notebook from the repo root
assert CHUNK_FILE.exists(), f"Put {CHUNK_FILE.name} in the same folder as this notebook (current folder: {Path.cwd()})"

raw = CHUNK_FILE.read_bytes()
chunks = json.loads(raw)
print(len(chunks), "chunks loaded:", dict(Counter(c["kind"] for c in chunks)))
print()
example = next(c for c in chunks if c["kind"] == "image")     # a sign with its "Meaning for the driver" line
print("Example chunk")
print("  topic     :", " > ".join(example["heading_path"]))
print("  applies to:", example["applies_to"])
print("  source    :", example["source"])
print("  text      :", example["text"][:300])

## Step 2. Load the embedding model

The embedding model turns text into vectors. This one (`bge-small-en-v1.5`) is free, runs on your own computer, and is about 130 MB, so the first run downloads it. Two pieces of text with similar meaning get similar vectors, even if they use different words.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "   # recommended for BGE, questions only

embedder = SentenceTransformer(EMBEDDING_MODEL)
demo = embedder.encode(["What is the blood alcohol limit?"], normalize_embeddings=True)
print("Each piece of text becomes a vector with", demo.shape[1], "numbers")

## Step 3. Store the vectors in Chroma (the vector database)

This embeds every chunk and saves the vectors in a local folder called `chroma_store`. It takes a minute or two the first time. After that, the saved index is reused, and it is **rebuilt automatically if the chunk file changes**, so you never search an out-of-date index.

In [ ]:
import chromadb

COLLECTION = "victoria_driving_kb"
fingerprint = hashlib.sha1(raw).hexdigest()          # changes whenever the chunk file changes
client = chromadb.PersistentClient(path="chroma_store")

col = None
try:
    existing = client.get_collection(COLLECTION)
    if (existing.metadata or {}).get("chunks_sha1") == fingerprint and existing.count() == len(chunks):
        col = existing
        print("Reusing the saved index (the chunk file has not changed).")
    else:
        client.delete_collection(COLLECTION)          # chunk file changed: rebuild
except Exception:
    pass

if col is None:
    col = client.create_collection(COLLECTION, metadata={"chunks_sha1": fingerprint})
    vectors = embedder.encode([c["embed_text"] for c in chunks], normalize_embeddings=True,
                              show_progress_bar=True).tolist()
    col.add(
        ids=[c["id"] for c in chunks],
        embeddings=vectors,
        documents=[c["text"] for c in chunks],
        metadatas=[{
            "heading": " > ".join(c["heading_path"]),
            "applies_to": c.get("applies_to") or "all drivers",
            "source": c.get("source") or c["doc_id"],
        } for c in chunks],
    )
    print("Built a new index with", col.count(), "chunks.")

## Step 4. Search: find the chunks that match a question

`retrieve` turns the question into a vector and returns the 5 closest chunks. `show_retrieval` prints them so you can check **what the chatbot would see**. This is the most useful cell for finding problems: if the right chunk is not in the results, the answer will be wrong, and the fix is in the knowledge base.

A very short follow-up such as "what about P plates?" is searched together with the previous question, so it keeps its context.

In [ ]:
TOP_K = 5

def search_query(question, history):
    earlier = [m["content"] for m in history if m["role"] == "user"]
    if earlier and len(question.split()) < 8:
        return earlier[-1] + " " + question
    return question

def retrieve(question, history=()):
    vec = embedder.encode([QUERY_PREFIX + search_query(question, history)], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=vec, n_results=TOP_K)
    return [{"text": d, "meta": m} for d, m in zip(res["documents"][0], res["metadatas"][0])]

def show_retrieval(question, n=3, history=()):
    print("Q:", question)
    for i, h in enumerate(retrieve(question, history)[:n], 1):
        preview = h["text"].replace("\n", " ")[:170]
        print(f"  {i}. [{h['meta']['heading']}]  ({h['meta']['source'][:45]})")
        print(f"     {preview}...")
    print()

show_retrieval("What is the blood alcohol limit for a full licence holder?")

## Step 5. Connect the free AI models and write the answer

This notebook uses two free providers: **Groq first, then Google Gemini as the backup**. If Groq is busy or out of quota, the notebook switches to Gemini by itself. You need **at least one key**; Groq alone is enough.

| Provider | Get a key | Free limits (checked October 2026; they change) |
|---|---|---|
| **Groq** (first) | console.groq.com/keys, no card | per model: 30 requests/min, 1,000/day, **8,000 tokens/min, 200,000 tokens/day**. That is about 5 answers a minute and 130 a day per model. The notebook tries three Groq models in turn |
| **Gemini** (backup) | aistudio.google.com/apikey | limits not published, so you only find out when you hit them |

**How to add keys:** set environment variables before launching Jupyter: `GROQ_API_KEY` and `GEMINI_API_KEY` (on Colab, store them as secrets with the same names). If no key is found, the next cell asks you to paste a Groq key. Model names are found automatically; to choose your own, set for example `GROQ_MODELS="openai/gpt-oss-120b"` or `GEMINI_MODELS="gemini-3.5-flash"`.

The prompt gives the model the retrieved chunks and these rules: answer **only** from them, say "I don't have that information" when the answer is not there, and when a rule depends on licence type and the user has not said which, give the **strictest** rule.

In [ ]:
import getpass, re, time
from collections import Counter
import openai
from IPython.display import display, Markdown

TEMPORARY = (429, 500, 502, 503, 504)      # busy, out of quota or briefly down
MAX_WAIT = 75                              # never wait longer than this many seconds for a busy model
PAUSE = 8                                  # seconds between questions in the long runs (Step 7 and Step 8); keeps Groq under 8,000 tokens/min

# Free providers, tried in this order. They all speak the same "OpenAI" protocol, so one library covers them.
# prefer = parts of model names to look for; each match becomes one link in the chain.
PROVIDERS = [
    dict(name="groq", key_env="GROQ_API_KEY", base="https://api.groq.com/openai/v1",
         prefer=["gpt-oss-120b", "gpt-oss-20b", "qwen3.8-27b"]),
    dict(name="gemini", key_env="GEMINI_API_KEY", base="https://generativelanguage.googleapis.com/v1beta/openai/",
         prefer=["gemini-3.5-flash"], first_only=True),
]

def _now():
    return time.time()

def _sleep(seconds):
    time.sleep(seconds)

def _key(p):
    key = os.environ.get(p["key_env"])
    if key:
        return key
    try:                                       # Google Colab secrets
        from google.colab import userdata
        return userdata.get(p["key_env"])
    except Exception:
        return None

def build_chain():
    chain = []
    for p in PROVIDERS:
        key = _key(p)
        if not key:
            continue
        client = openai.OpenAI(api_key=key, base_url=p["base"], max_retries=0, timeout=60)
        override = os.environ.get(p["name"].upper() + "_MODELS")      # e.g. GROQ_MODELS="openai/gpt-oss-120b"
        try:
            if override:
                models = [m.strip() for m in override.split(",") if m.strip()]
            else:
                ids = [m.id.replace("models/", "", 1) for m in client.models.list()]
                models = []
                for want in p["prefer"]:
                    hit = next((i for i in ids if want in i), None)
                    if hit and hit not in models:
                        models.append(hit)
                    if hit and p.get("first_only"):
                        break
        except openai.APIConnectionError:
            print(f"  {p['name']}: could not connect, skipping")
            continue
        except openai.APIStatusError as e:
            print(f"  {p['name']}: could not list models (error {e.status_code}). Check the key. Skipping")
            continue
        for m in models:
            extra = {"reasoning_effort": "low"} if p["name"] == "groq" and "gpt-oss" in m else {}
            chain.append(dict(name=p["name"], model=m, client=client, cool_until=0.0, off=None, extra=extra))
    return chain

def start_chain():
    chain = build_chain()
    if not chain:
        key = getpass.getpass("No free AI key found. Paste a Groq key (free, no card: console.groq.com/keys), or press Enter to stop: ").strip()
        if key:
            os.environ["GROQ_API_KEY"] = key
            chain = build_chain()
    if not chain:
        raise RuntimeError("No AI model is available. Get a free Groq key at https://console.groq.com/keys, "
                           "set GROQ_API_KEY (or paste it when asked) and run this cell again.")
    print("Free AI models, in the order they are tried:")
    for i, s in enumerate(chain, 1):
        print(f"  {i}. {s['name']} / {s['model']}")
    return chain

chain = start_chain()

PROMPT = """You are a helpful assistant answering international students' questions about driving \
in Victoria, Australia, especially for people driving on an overseas licence.

Rules you must follow:
1. Answer ONLY using the context below. Do not use outside knowledge.
2. If the answer is not in the context, say: "I don't have that information in my knowledge base. \
Please check the official VicRoads or Transport Victoria website."
3. If a rule depends on licence type (for example alcohol limits or phone use) and the user has not \
said which licence they hold, give the STRICTEST rule as the safe default and say the exact rule \
depends on their licence type.
4. Say which source each rule comes from (shown as "Source:" in the context).
5. Keep answers short and in plain language. Many users are not native English speakers.
6. If the context marks something as [WARNING], make clear it is a firm rule, not a tip.
7. If the context contains conflicting information, prefer the more recent source and say the user \
should check the official website.

Context:
{context}
{history}
Question: {question}

Answer:"""

def build_prompt(question, hits, history):
    context = "\n\n---\n\n".join(
        f"[Source: {h['meta']['source']} | Applies to: {h['meta']['applies_to']} | Topic: {h['meta']['heading']}]\n{h['text']}"
        for h in hits)
    recent = history[-6:]
    hist = ""
    if recent:
        hist = "\nConversation so far:\n" + "\n".join(
            f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content'][:600]}" for m in recent) + "\n"
    return PROMPT.format(context=context, history=hist, question=question)

class AllBusy(Exception):
    pass

provider_counts = Counter()

def _retry_after(e):
    try:
        return float(e.response.headers.get("retry-after"))
    except Exception:
        return None

def ask_llm(prompt):
    """Try each model in turn. Returns (answer, 'provider/model')."""
    last = "no model could be used"
    for attempt in range(4):
        waiting = False
        for s in chain:
            if s["off"]:
                continue
            if s["cool_until"] > _now():
                waiting = True
                continue
            for extra in ([s["extra"], {}] if s["extra"] else [{}]):
                try:
                    r = s["client"].chat.completions.create(
                        model=s["model"], temperature=0.2, extra_body=extra or None,
                        messages=[{"role": "user", "content": prompt}])
                    text = (r.choices[0].message.content or "").strip()
                    if not text:
                        last = f"{s['name']} gave an empty answer"
                        break
                    s["extra"] = extra
                    who = s["name"] + "/" + s["model"]
                    provider_counts[who] += 1
                    return text, who
                except openai.APIStatusError as e:
                    code = e.status_code
                    last = f"{s['name']}/{s['model']}: error {code}"
                    if code == 400 and extra:
                        continue                          # this model dislikes the extra setting: retry without it
                    if code in TEMPORARY:
                        wait = _retry_after(e) or 20
                        s["cool_until"] = _now() + wait
                        waiting = True
                        if wait > 900:
                            print(f"  {s['name']}/{s['model']}: quota used up for now, skipping it")
                    elif code in (401, 403, 404):
                        s["off"] = f"error {code}"
                        print(f"  {s['name']}/{s['model']}: error {code} (bad key or model name). Not using it again")
                    break
                except openai.APIConnectionError:
                    s["cool_until"] = _now() + 30
                    last = f"{s['name']}: no connection"
                    waiting = True
                    break
        if not waiting:
            break
        soonest = min(s["cool_until"] for s in chain if not s["off"]) - _now()
        if soonest > MAX_WAIT or attempt == 3:
            break
        print(f"All free models are busy. Waiting {max(soonest, 1):.0f} seconds...")
        _sleep(max(soonest, 1) + 0.5)
    raise AllBusy(last)

def friendly_error(e):
    if isinstance(e, AllBusy):
        return (f"All the free AI models are busy or out of quota right now ({e}). This is temporary: wait a minute and "
                f"ask again, or, if it keeps happening, today's free quota is probably used up.")
    return f"Sorry, something went wrong while getting an answer ({type(e).__name__}: {e})."

history = []          # the conversation so far

def reset_chat():
    history.clear()

def ask(question, show_sources=True):
    hits = retrieve(question, history)
    who = ""
    try:
        answer, who = ask_llm(build_prompt(question, hits, history))
    except Exception as e:
        answer = friendly_error(e)
    history.append({"role": "user", "content": question})
    history.append({"role": "assistant", "content": answer})
    display(Markdown(f"**Q: {question}**\n\n{answer}" + (f"\n\n*answered by {who}*" if who else "")))
    if show_sources:
        print("Sources used:")
        for h in hits:
            print("  -", h["meta"]["heading"], "|", h["meta"]["source"][:60])
    return answer

ask("What is the blood alcohol limit if I hold an overseas licence?")

## Step 6. Chat box

Type a question and press **Ask**. The chat remembers the conversation, so you can ask follow-ups. Run `reset_chat()` in any cell to start fresh.

In [ ]:
import ipywidgets as widgets

box = widgets.Text(placeholder="Ask about driving in Victoria...", layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Ask", button_style="primary")
out = widgets.Output()

def on_send(_):
    q = box.value.strip()
    if not q:
        return
    box.value = ""
    with out:
        ask(q)

send.on_click(on_send)
display(widgets.HBox([box, send]), out)

## Step 7. Test it with real questions

Write down every wrong or vague answer. Run the **first cell** (search only, free and fast) to check that the right chunk is being found. Run the **second cell** to see the full answers. The list includes the known weak spots.

In [ ]:
TEST_QUESTIONS = [
    "I'm 27. How long must I hold a learner permit before the probationary test?",   # known weak spot
    "Does a licence from India need a driving test to convert?",                      # known weak spot
    "Can I have one beer if I drive on an overseas licence?",
    "How long can I drive in Victoria on my overseas licence before I must convert it?",
    "What does the white T on a traffic light mean?",
    "What is a hook turn and how do I do it?",
    "What is the fine for speeding 15 km/h over the limit?",
    "Who gives way at a roundabout?",
    "Can I use my phone for GPS while driving?",
    "What is the speed limit near a school?",
    "Can I overtake on a single continuous white line?",
    "What do I do if the traffic lights are not working?",
    "Can I park within 20 metres of a bus stop?",
    "What should I do if I miss my freeway exit?",
    "What should I do if I see a kangaroo on the road?",                              # not in the knowledge base
]

for q in TEST_QUESTIONS:
    show_retrieval(q, n=2)

In [ ]:
for q in TEST_QUESTIONS:
    reset_chat()
    ask(q, show_sources=False)
    print("-" * 70)
    _sleep(PAUSE)

## Step 8. Evaluate: score the search and the answers

The 47 questions in `evaluation_questions.json` are written the way students ask, and each one lists the fact a correct answer needs. The expected facts were checked against the original handbook where it covers them.

**8a. Score the search (no API key needed).** For each question, is a chunk that contains the right fact among the 5 the chatbot sees? If not, the answer will probably be wrong, whatever Gemini does.

**8b. Score the answers (uses the free AI models, about 8 minutes with one Groq key because the questions are paced to stay under the free token limits).** Each answer is checked for the facts it must contain and the claims it must not make. Questions that are not in the knowledge base must be answered with "I don't have that information". This is an automatic first pass: read the failures yourself, because a different wording can fail the check and a correct-looking answer can still be wrong.

In [ ]:
import re, time

EVAL_FILE = Path("data/evaluation_questions.json")
assert EVAL_FILE.exists(), f"Put {EVAL_FILE.name} in the same folder as this notebook"
eval_qs = json.loads(EVAL_FILE.read_text())

def chunk_matches(q, hit):
    text = hit["meta"]["heading"] + "\n" + hit["text"]
    return any(all(re.search(g, text, re.I) for g in facts) for facts in [q["gold"]] + q.get("alt_gold", []))

search_rows = []
for q in eval_qs:
    if q["retrieval_eval"]:
        hits = retrieve(q["question"])
        rank = next((i for i, h in enumerate(hits, 1) if chunk_matches(q, h)), None)
        search_rows.append((q, rank, hits))

n = len(search_rows)
for k in (1, 3, 5):
    got = sum(1 for _, r, _ in search_rows if r and r <= k)
    print(f"Right chunk in the top {k}: {got} of {n} ({got / n:.0%})")

print("\nBy category (right chunk in the top 5):")
for cat in dict.fromkeys(q["category"] for q, _, _ in search_rows):
    rows = [r for q, r, _ in search_rows if q["category"] == cat]
    print(f"  {cat:24s} {sum(1 for r in rows if r)} of {len(rows)}")

print("\nMisses (the right chunk is not in the top 5):")
for q, r, hits in search_rows:
    if not r:
        print(f"  #{q['id']} {q['question']}\n      ranked first: {hits[0]['meta']['heading'][:90]}")

In [ ]:
provider_counts.clear()

def score_answer(q, answer):
    problems = [f"missing: {p}" for p in q["answer_must"] if not re.search(p, answer, re.I)]
    problems += [f"should not say: {p}" for p in q["answer_must_not"] if re.search(p, answer, re.I)]
    return problems

def answer_only(question):
    try:
        return ask_llm(build_prompt(question, retrieve(question), []))[0]
    except Exception as e:
        return "ERROR: " + friendly_error(e)

answer_rows = []
for q in eval_qs:
    a = answer_only(q["question"])
    answer_rows.append((q, a, score_answer(q, a)))
    _sleep(PAUSE)

print("Answers came from:", dict(provider_counts))
passed = sum(1 for _, _, p in answer_rows if not p)
print(f"Answers that pass the automatic check: {passed} of {len(answer_rows)} ({passed / len(answer_rows):.0%})\n")
for cat in dict.fromkeys(q["category"] for q, _, _ in answer_rows):
    rows = [p for q, _, p in answer_rows if q["category"] == cat]
    print(f"  {cat:24s} {sum(1 for p in rows if not p)} of {len(rows)}")
print("\nFailures to read by hand:")
for q, a, p in answer_rows:
    if p:
        print(f"\n#{q['id']} {q['question']}\n  problems: {'; '.join(p)}\n  answer: {a[:400]}")

## Step 9. Fixing wrong answers

Look at **Step 4** (search) or the misses in **Step 8a** first for each wrong answer:

| What you see | What it means | What to do |
|---|---|---|
| The right chunk is **not** in the results | The fact is missing, or buried in a long chunk | Edit the Word file: add the fact, or add a question-style line next to it (for example "How long must I hold a learner permit if I'm 27?") |
| The right chunk **is** in the results but the answer is wrong | The wording is unclear, or two chunks disagree | Rewrite the chunk so it stands alone, and fix any conflict |
| It answers something that is not in the knowledge base | The prompt rules are not strict enough | Tighten the rules in the `PROMPT` text in Step 5 |

After editing the Word file, re-chunk it (with `extract_and_chunk.py`) to get a new `integrated_kb_chunks.json`, then re-run from **Step 1**. Step 3 rebuilds the index automatically.